In [2]:
from IPython.display import display, Markdown

In [3]:
### For printing strings with bigger font
def printmd(string):
    display(Markdown(string))

## Functions to check types of objects

In [4]:
## Return True if obejct is python number class
from numbers import Number
def is_number(ob):
    return isinstance(ob, Number)

## Return True if obejct is scalar object in verify or python number class
def is_scalar(ob):
    try:
        return ob.is_scalar
    except:
        return False
def is_scalar_or_number(ob):
    try:
        return ob.is_scalar
    except:
        return isinstance(ob, Number)

## Return True if obejct is vector object in verify
def is_vector(ob):
    try:
        return ob.is_vector
    except:
        return False

## Return True if obejct is VerifyMatrix object in verify
def is_matrix(ob):
    try:
        return ob.is_Matrix
    except:
        return False

## Class for sorting and organizing args

In [5]:
class ArgsOrganizer():
    ## Sort args with str of each arg
    def sort_args(self, args):
        str_dict = {}
        list_to_sort = []
        for i in range(len(args)):
            str_dict[str(args[i])] = i
            list_to_sort.append(str(args[i]))
        sorted_list = sorted( list_to_sort )
        for j in range(len(sorted_list)):
            sorted_list[j] = args[ str_dict[ sorted_list[j] ] ]
        return tuple(sorted_list)

########------------------------------------------ Orgnizers for args for each classes ---------------------------------------#########

    ## For ScalarMul class, a*a becomes ScalarPow(a,2)
    def organize_ScalarMul_args(self, args):
        inde_scalars = set()
        new_args_dict = {}
        for arg in args:
            inde_scalar = arg.base
            if inde_scalar in inde_scalars:
                new_args_dict[inde_scalar] = new_args_dict[inde_scalar] + arg.exponent
            else:
                inde_scalars.add(inde_scalar)
                new_args_dict[inde_scalar] = arg.exponent
        # gather new_args
        new_args = ()
        for arg in new_args_dict:
            # if new_arg==1, skip it
            if not new_args_dict[arg] == 0:
                new_args += ( ScalarPow( arg, new_args_dict[arg] ), )
        return self.sort_args( new_args )

    ## For ScalarAdd class, a+a becomes ScalarMul(2,a,1,1)
    def organize_ScalarAdd_without_IP_FV(self, constant, args):
        inde_args = set()
        new_args_dict = {}
        for arg in args:
            if arg ==0:  #XXX 임시방편
                continue
            if type(arg)==ScalarMul:
                inde_arg = arg.args
            else:
                inde_arg = (arg,)   ## for Scalar, ScalarPow : a should be identified with (a,)
            if inde_arg in inde_args:     # if inde_arg is already in inde_args, add num_coeffi to previous num_coeffi
                new_args_dict[inde_arg] = new_args_dict[inde_arg] + arg.num_coeffi
            else:   # if inde_arg is not in inde_args, add it inde_args and initialize num_coeffi
                inde_args.add( inde_arg )
                new_args_dict[inde_arg] = arg.num_coeffi
        # gather new_constant and new_args
        new_constant = constant
        new_args = ()
        for arg in new_args_dict:
            new_arg_num_coeffi = new_args_dict[arg]
            arg_to_append = ScalarMul( new_arg_num_coeffi, arg, 1, 1 )
            if is_number( arg_to_append ):
                new_constant += arg_to_append
            else:
                new_args += ( arg_to_append, )
        return new_constant, self.sort_args( new_args )

    ## For ScalarAdd class, IP(v,v)+IP(v,v) becomes ScalarMul(2,1,IP(v,v),1)
    def organize_ScalarAdd_with_IP_FV(self, args, inde_IPs, inde_FVs):
        non_IP_FV_terms = 0
        new_IPs_dict = {}
        new_FVs_dict = {}
        for IP in inde_IPs:
            new_IPs_dict[IP] = 0
        for FV in inde_FVs:
            new_FVs_dict[FV] = 0
        for arg in args:
            if arg==0:  #XXX 임시방편
                continue
            if arg.IP_term in inde_IPs:
                if type(arg)==ScalarMul:
                    new_IPs_dict[arg.IP_term] = new_IPs_dict[arg.IP_term] + ScalarMul( arg.num_coeffi,arg.args, 1, 1)
                else:   # if arg contains IP_term but not is ScalarMul type, it is IP
                    new_IPs_dict[arg.IP_term] = new_IPs_dict[arg.IP_term] + 1
            elif arg.FV_term in inde_FVs:
                if type(arg)==ScalarMul:
                    new_FVs_dict[arg.FV_term] = new_FVs_dict[arg.FV_term] + ScalarMul( arg.num_coeffi,arg.args, 1, 1)
                else:   # if arg contains FV_term but not is ScalarMul type, it is FV
                    new_FVs_dict[arg.FV_term] = new_FVs_dict[arg.FV_term] + 1
            else:
                non_IP_FV_terms = non_IP_FV_terms + arg
        # Initiallize new_args with non_IP_FV_terms
        if type(non_IP_FV_terms)==ScalarAdd:
            new_args = non_IP_FV_terms.args
        elif non_IP_FV_terms==0:
            new_args = ()
        else:
            new_args = ( non_IP_FV_terms, )
        # gather IP and FV terms in new_args
        for IP in new_IPs_dict:
            new_args += ( new_IPs_dict[IP] * IP, )
        for FV in new_FVs_dict:
            new_args += ( new_FVs_dict[FV] * FV, )
        return self.sort_args( new_args )

    ## For VectorAdd class, v+v becomes VectorMul(2,v)
    def organize_VectorAdd_args(self, args):
        inde_vecs = set()
        new_args_dict = {}
        for arg in args:
            inde_vec = arg.vector
            if inde_vec in inde_vecs:
                new_args_dict[inde_vec] = new_args_dict[inde_vec] + arg.coeffi
            else:
                inde_vecs.add(inde_vec)
                new_args_dict[inde_vec] = arg.coeffi
        # gather new_args
        new_args = ()
        for arg in new_args_dict:
            # if coefficient is zero, skip it
            if not new_args_dict[arg] == 0:
                new_args += ( VectorMul( new_args_dict[arg], arg ), )
        return self.sort_args( new_args )

---
# Classes of materials to express Algorithm :

### String expression for operations

In [6]:
MulScalVecStr = " * "
MulScalScalStr = " * "
PowStr = "**"

# Inequality

In [7]:
class Ineq():
    def __init__(self, lhs, rhs):
        self.args = (lhs, rhs)
        self.truth = 'unknown'

    def __str__(self):
        s = str(self.args[0]) + " <= " + str(self.args[1])
        return s

    def __repr__(self):
        s = repr(self.args[0]) + " \\le " + repr(self.args[1])
        return s

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)

    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        result = set()
        for arg in self.args:
            if not is_number( arg ):
                result = result.union( arg.get_inde_Point() )
        return result

    ## Method to get 'free variables' for IP as set
    def get_inde_IP(self):
        result = set()
        for arg in self.args:
            if not is_number( arg ):
                result = result.union( arg.get_inde_IP() )
        return result

    ## Method to get 'free variables' for IP as set
    def get_inde_FV(self):
        result = set()
        for arg in self.args:
            if not is_number( arg ):
                result = result.union( arg.get_inde_FV() )
        return result

    ## Get 'free variable' for scalars
    def get_inde_scalar(self):
        result = set()
        for arg in self.args:
            if not is_number( arg ):
                result = result.union( arg.get_inde_FV() )
        return result

# Basic object for verify objects

In [239]:
## The basic obejct for objects in Verify
class VerifyObject():
    name = ""
    is_scalar = False
    is_vector = False
    is_IP = False
    is_FV = False
    is_Matrix = False

    ## Big case division is done here
    def __add__(self, other):
        if is_vector(self) and is_vector(other):
            return self.vec_vec_add(other)
        elif is_scalar_or_number(self) and is_scalar_or_number(other) :
            return self.scal_scal_add(other)
        if is_matrix(self) and is_matrix(other):
            return self.__add__(other)
        else:
            raise TypeError( "Can't add " +  str(self) + ' of type ' + str(type(self)) + " with " + str(other) + ' of type ' + str(type(other)) )

    __radd__ = __add__

    def __sub__(self, other):
        return self.__add__(-other)

    def __rsub__(self, other):
        return (-self).__add__(other)

    ## Big case division is done here
    def __mul__(self, other):
        ## both are scalars
        if is_scalar_or_number(self) and is_scalar_or_number(other):
            if is_scalar(self):
                return self.scal_scal_mul(other)
            elif is_scalar(other):
                return other.scal_scal_mul(self)
        ## one scalar, one vector
        elif is_vector(self) and is_scalar_or_number(other):
            return self.scal_vec_mul(other)
        elif is_scalar_or_number(self) and is_vector(other):
            return other.scal_vec_mul(self)
        ## both are vectors
        elif is_vector(self) and is_vector(other):
            raise TypeError( "Can't multiply vector with vector" )
        ## When eitherone is matrix
        elif is_matrix(self):
            return self.__mul__(other)
        elif is_matrix(other):
            return other.__mul__(self)


    __rmul__ = __mul__

    def __pos__(self):
        return self

    def __neg__(self):
        return self.__mul__(-1)

    def __truediv__(self, other):
        if is_scalar_or_number(other):
            return self.__mul__( ScalarPow(other, -1) )
        else:
            raise TypeError( "Can't divide with " + str(type(other)) )

    def __rtruediv__(self, other):
        if is_scalar_or_number(self):
            return ScalarPow(self, -1).__mul__( other )
        else:
            raise TypeError( "Can't divide with " + str(type(self)) )

    def __str__(self):
        return self.name

    def __repr__(self):
        s = str(self)
        s = s.replace(PowStr, '^')
        s = s.replace(MulScalVecStr, ' ')
        s = s.replace(MulScalScalStr, ' ')
        return s

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)


    def __ge__(self, other):
        return Ineq(self, other)

    def __le__(self, other):
        return Ineq(other, self)

    ## We may want to think objects with same type and expression as same obejct
    def __eq__(self, other):
        if type(self)==type(other):
            return str(self)==str(other)
        else:
            return False

    ## To gather objects in set, __hash__ is required
    def __hash__(self):
        return hash( str(type(self)) + str(self) )

    ## This checks if the obejct is 'zero' for each class
    def is_zero(self):
        if self == 0 or self == ZeroVector:
          return True
        expanded_input = self.expand()
        if is_scalar_or_number(expanded_input):
            return expanded_input == 0
        elif is_vector(expanded_input):
           return expanded_input.equals(ZeroVector)
        elif is_matrix(expanded_input):
            for row_list in expanded_input.mat:
                for entry in row_list:
                    if is_scalar_or_number(entry):
                        if entry!=0:
                            return False
                    elif not entry.is_zero():
                        return False
            return True

    ## When we check if an object is zero, we expand all terms by distribution law
    def expand(self):
        return self

    ## XXX repeat expand until it stops to change
    def simplify(self):
        result = self
        while True:
            try:
                new_result = result.expand()
                if result != new_result:
                    result = new_result
                else:
                    break
            except:
                break
        return result

    def substitute(self, equality):
        # try:
        #   return equality[self]
        # except:
        if equality.lhs == self:
            return equality.rhs
          # elif equality.rhs == self: return equality.lhs.  <- First leave this part to prevent unwanted way of substitution
        else:
            return self

In [240]:
## Parent class for terms with addition, not used yet
class VerifyAdd(VerifyObject):
    pass

# Scalars

In [241]:
class Scalar(VerifyObject):
    is_scalar = True
    is_real = True
    ## initializing defaul setting
    is_integer = False
    is_positive = False
    is_negative = False
    is_nonnegative = False
    num_coeffi = 1
    exponent = 1
    IP_term = 1
    FV_term = 1

    def __init__(self, name, **kwargs):
        self.name = name
        self.base = self

    ## Addition between scalars
    def scal_scal_add(self, other):
        if self==0:
            return other
        if other==0:
            return self
        if type(self)==ScalarAdd and type(other)==ScalarAdd:
            return ScalarAdd( self.constant + other.constant , self.args + other.args )
        elif type(self)==ScalarAdd:
            if is_number(other):
                return ScalarAdd( self.constant + other, self.args )
            else:
                return ScalarAdd( self.constant, self.args + (other,) )
        elif type(other)==ScalarAdd:
            if is_number(self):
                return ScalarAdd( other.constant + self, other.args )
            else:
                return ScalarAdd( other.constant, other.args + (self,) )
        else:
            if is_number(self):
                return ScalarAdd( self, (other,) )
            elif is_number(other):
                return ScalarAdd( other, (self,) )
            else:
                return ScalarAdd( 0, (self , other) )

    ## Multiplication between scalars
    def scal_scal_mul(self, other):
        if self==0 or other==0:
            return 0
        if self==1:
            return other
        if other==1:
            return self
        parsed_self = self.parse_to_tuple(self)
        parsed_other = self.parse_to_tuple(other)
        result = ScalarMul( parsed_self[0] * parsed_other[0] ,
                            parsed_self[1] + parsed_other[1]  ,
                            self.IP_Mul( parsed_self[2], parsed_other[2] ) ,
                            self.FV_Mul( parsed_self[3], parsed_other[3] ) )
        return result

    ## For operations, parse scalar object(including python numbers) to tuple
    def parse_to_tuple(self, ob):
        if type(ob)==ScalarMul:
            return (ob.num_coeffi, ob.args, ob.IP_term, ob.FV_term)
        elif type(ob)==Scalar or type(ob)==ScalarPow:
            return (1, (ob,), 1, 1)
        elif is_number(ob):
            return (ob, (), 1, 1)
        elif type(ob)==IP:
            return (1, (), ob, 1)
        elif type(ob)==FV:
            return (1, (), 1, ob)
        else:
            return (1, (ob,) , 1, 1)

    ## For now, we don't consider IP*IP but later may not
    def IP_Mul(self, IP1, IP2):
        if IP1==1:
            return IP2
        elif IP2==1:
            return IP1
        else:
            raise TypeError( "For now, can't multiply " + str(type(self)) + " with " + str(type(other)) )

    ## For now, we don't consider FV*FV but later may not
    def FV_Mul(self, FV1, FV2):
        if FV1==1:
            return FV2
        elif FV2==1:
            return FV1
        else:
            raise TypeError( "For now, can't multiply " + str(type(self)) + " with " + str(type(other)) )

    ## Get 'free variable' for scalars for organization
    def get_inde_scalar(self):
        return { self }

    ## Get 'free variable' of IP terms
    def get_inde_IP(self):
        return set()

    ## Get 'free variable' of FV terms
    def get_inde_FV(self):
        return set()

    ## when self!=ScalarAdd, distribution_law returns basic multiplication
    def distribution_law(self, coeffi):
        return self * coeffi

    def __pow__(self, other):
        return ScalarPow(self, other)

    # def sqrt(self):
    #     if le(0,self.base) in Prop:
    #         result = ScalarPow(self.base,1/2)
    #         Prop.add(Equal(result*result, self.base))
    #         return result
    #     else: raise ValueError("should be nonnegative inside a squareroot")

In [247]:
class ScalarMul(Scalar):
    def __init__(self, num_coeffi, args, IP_term, FV_term):
        self.num_coeffi = num_coeffi
        self.args = self.organize_args(args)
        self.IP_term = IP_term
        self.FV_term = FV_term

    ## For each cases, other types of objects shoul be returned
    def __new__(cls, num_coeffi, args, IP_term, FV_term):
        organize_args = cls.organize_args(cls, args)
        no_coeffi = num_coeffi==1
        no_args = organize_args==()
        no_IP = IP_term==1
        no_FV = FV_term==1
        if num_coeffi==0:
            return 0
        if (not no_IP) and (not no_FV):
            raise TypeError( "For now, can't multiply" + str(type(IP_term)) + " with " + str(type(FV_term)) )
        if no_args and no_IP and no_FV:
            return num_coeffi             # python number
        elif no_coeffi and no_args and no_IP:
            return FV_term
        elif no_coeffi and no_args and no_FV:
            return IP_term
        elif no_coeffi and no_IP and no_FV:
            if len(organize_args)==1:
                return organize_args[0]
            else:
                return super().__new__(cls)
        else:
            return super().__new__(cls)

    def __str__(self):
        s = ""
        if self.num_coeffi==-1:
            s += "-" #+ MulScalScalStr
        elif not self.num_coeffi==1:
            s += str(self.num_coeffi) + MulScalScalStr
        if not self.args == ():
            for i in range( len(self.args)-1 ):
                if type( self.args[i] ) == ScalarAdd:
                    s += "( " + str( self.args[i] ) + " )" + MulScalScalStr
                else:
                    s += str( self.args[i] ) + MulScalScalStr
            if type( self.args[-1] ) == ScalarAdd:
                s += "( " + str( self.args[-1] ) + " )"
            else:
                s += str( self.args[-1] )
        if not self.FV_term==1:
            s += MulScalScalStr + str(self.FV_term)
        if not self.IP_term==1:
            s += MulScalScalStr + str(self.IP_term)
        return s

    def __pow__(self, other):
        powered_num_coeffi = self.num_coeffi**other # should be changed later
        powered_args_list = []
        for arg in self.args:
            powered_args_list.append(arg**other)
        powered_args = tuple(powered_args_list)
        powered_IP_term = self.IP_term**other
        powered_FV_term = self.FV_term**other
        return ScalarMul(powered_num_coeffi, powered_args, powered_IP_term, powered_FV_term)
    
    ## Get 'free variable' for scalars for organization
    def get_inde_scalar(self):
        inde_scalars = set()
        for arg in self.args:
            inde_scalars.union( arg.get_inde_scalar() )
        return inde_scalars

    ## Organize exponents, ex) if args=(a,a), it becomse args=(ScalarPow(a,2),)
    def organize_args(self, args):
        og = ArgsOrganizer()
        return og.organize_ScalarMul_args(args)

    ## Get 'free variable' of IP terms
    def get_inde_IP(self):
        if self.IP_term==1:
            return set()
        else:
            return { self.IP_term }

    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        if self.IP_term==1:
            return set()
        else:
            return self.IP_term.get_inde_Point()

    ## Get 'free variable' of FV terms
    def get_inde_FV(self):
        if self.FV_term==1:
            return set()
        else:
            return { self.FV_term }

    ## For example, 3a(b+c)d <v,w> = 3abd <v,w> + 3acd <v,w>
    def expand(self):
        result = 0
        if self.IP_term==1: # XXX 임시방편, inner product 항이 있으면 전개하라는 코드
            coeffi_for_args = self.num_coeffi * self.IP_term * self.FV_term
        else:
            coeffi_for_args = self.num_coeffi * self.IP_term.expand() * self.FV_term
        expanded_args = self.args_expansion( self.args )
        if expanded_args==():
            return coeffi_for_args
        else:
            if expanded_args==0:  #XXX 임시방편
                return 0
            return expanded_args.distribution_law(coeffi_for_args)


    ## For exmaple, a(b+c)d = abd + acd
    def args_expansion(self, args_to_expand):
        if len(args_to_expand)==0:
            return args_to_expand
        if len(args_to_expand)==1:
            return args_to_expand[0].expand()
        else:
            result = 0
            expanded_term = self.args_expansion(args_to_expand[1:])
            if not type(expanded_term)==ScalarAdd:
                return args_to_expand[0].expand().distribution_law(expanded_term)
            for expaned_arg in expanded_term.args:
                result = result + args_to_expand[0].expand().distribution_law(expaned_arg)
            result = result + args_to_expand[0].distribution_law(expanded_term.constant)
            return result.expand()

    def substitute(self, sub_dict):
        result = self.num_coeffi
        for arg in self.args:
            result *= arg.substitute(sub_dict)
        try:
            result *= self.IP_term.substitute(sub_dict)
        except:
            result *= self.IP_term
        try:
            result *= self.FV_term.substitute(sub_dict)
        except:
            result *= self.FV_term
        return result

In [248]:
class ScalarAdd(Scalar):
    def __init__(self, constant, args):
        self.constant, self.args = self.organize_args(constant, args)
        self.base = self

    ## For each cases, other types of objects shoul be returned
    def __new__(cls, constant, args):
        if args==():
             return constant
        if constant==0 and len(args)==1:
             return args[0]
        new_constant, organized_args = cls.organize_args(cls, constant, args)
        if organized_args==():
            return new_constant
        if new_constant==0 and len(organized_args)==1:
            return organized_args[0]
        return super().__new__(cls)

    def __str__(self):
        s = ""
        if not self.constant==0:
            s += str(self.constant)
            if self.args[0].num_coeffi<0:
                s += str(self.args[0])
            else:
                s += "+" + str(self.args[0])
        else:
             s += str(self.args[0])
        # len(self.args)>0 since if not, it is constant so python number object
        # If num_coeffi<0, erase "+" to avoid representation like a+ -b
        if len(self.args)>1:
            for i in range( 1, len(self.args) ):
                try: # XXX 임시방편
                    if self.args[i].num_coeffi<0:
                        s += str( self.args[i] )
                    else:
                        s += "+" + str( self.args[i] )
                except:
                        s += "+" + str( self.args[i] )
        return s

    ## we dont want to see a-(b+c) as a+(-1)(b+c)
    def __neg__(self):
        new_args = ()
        for arg in self.args:
            new_args += (-arg,)
        return ScalarAdd(-self.constant, new_args)
    
    def __pow__(self, other):
        return ScalarPow(self, other)


    ## Get 'free variable' for scalars for organization
    def get_inde_scalar(self):
        inde_scalars = set()
        for arg in self.args:
            inde_scalars.union( arg.get_inde_scalar() )
        return inde_scalars

    ## Get all 'independent IP' terms exist in Add
    def get_inde_IP(self):
        inde_IPs = set()
        for arg in self.args:
            inde_IPs = inde_IPs.union( arg.get_inde_IP() )
        return inde_IPs


    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        inde_Points = set()
        for arg in self.args:
            inde_Points = inde_Points.union( arg.get_inde_Point() )
        return inde_Points

    ## Get all 'independent FV' terms exist in Add
    def get_inde_FV(self):
        inde_FVs = set()
        for arg in self.args:
            inde_FVs = inde_FVs.union( arg.get_inde_FV() )
        return inde_FVs

    ## Organize as linear combination, ex) change a+a to ScalarMul(2, (a,), 1, 1) = 2a
    def organize_args(self, constant, args):
        og = ArgsOrganizer()

        ## Gather IP and FV terms in args
        inde_IPs, inde_FVs = set(), set()
        for arg in args:
            if arg !=0:  #XXX 임시방편
                inde_IPs = inde_IPs.union( arg.get_inde_IP() )
                inde_FVs = inde_FVs.union( arg.get_inde_FV() )

        ## Treat each cases differently : with/without IP or FV terms
        if inde_IPs==set() and inde_FVs==set():
            return og.organize_ScalarAdd_without_IP_FV(constant, args)
        else:
            return constant, og.organize_ScalarAdd_with_IP_FV(args, inde_IPs, inde_FVs)

    ## ex) 1 + a + (b+c)d = 1 + a + bd + cd
    def expand(self):
        result = 0
        for arg in self.args:
            try: # 임시방편
                result = result + arg.expand()
            except:
                result = result + arg
        result = result + self.constant
        return result

    ## if self = a+b+c, coeffi=d, then return ad + bd + cd
    def distribution_law(self, coeffi):
        result = 0
        result = result + self.constant * coeffi
        for arg in self.args:
            result = result + arg * coeffi
        return result

    def substitute(self, sub_dict):
        result = self.constant
        for arg in self.args:
            result += arg.substitute(sub_dict)
        return result

In [313]:
class ScalarPow(Scalar):
    def __init__(self, base, exponent):
        self.base = base
        self.exponent = exponent

    def __new__(cls, base, exponent):
        if exponent==0:
            return 1
        if exponent==1:
            return base
        if is_number(base) and is_number(exponent):
            return base**exponent
        return super().__new__(cls)

    def __str__(self):
        if type(self.base)==ScalarAdd:
            return "(" + str(self.base) + ")"+ "**{" + str(self.exponent) + "}"
        return str(self.base) + "**{" + str(self.exponent) + "}"

    ## (x**a)**b = x**(ab)
    def __pow__(self, index):
        return ScalarPow(self.base, self.exponent * index )

    ## Get 'free variable' for scalars for organization
    def get_inde_scalar(self):
        return { self.base }

    ## For exmaple, (a+b)^2= a^2 + 2ab + b^2
    def expand(self):
        expanded_base = self.base.expand()
        if type( expanded_base ) != ScalarAdd:
            return self
        exp = self.exponent
        if is_number(self.exponent):
            if int(exp)==exp and exp>0:
                return self.poly_expansion(expanded_base, exp)
        return self

    ## expand ( a_1 + ... + a_n)^n
    def poly_expansion(self, ScalAdd, n):
        if n==1:
            return ScalAdd
        else:
            result = 0
            expanded_term = self.poly_expansion(ScalAdd, n-1)
            for expaned_arg in expanded_term.args:
                result = result + ScalAdd.distribution_law(expaned_arg)
            result = result + ScalAdd.distribution_law(expanded_term.constant)
            return result




    def substitute(self, sub_dict):
        try:
            base = self.base.substitute(sub_dict)
        except:
            base = self.base
        try:
            exponent = self.exponent.substitute(sub_dict)
        except:
            exponent = self.exponent
        return ScalarPow(base,exponent)

In [314]:
a = Scalar('a')
b = Scalar('b')

(a*b)**(1/2)-a**(1/2)*b**(1/2)

0

# Vectors

In [261]:
class Vector(VerifyObject):
    is_vector = True
    coeffi = 1

    def __init__(self, name):
        self.name = name
        self.vector = self

    ## Addition between two vectors
    def vec_vec_add(self, other):
        if self == ZeroVector:
            return other
        if other == ZeroVector:
            return self
        if type(self)==VectorAdd:
            if type(other)==VectorAdd:
                return VectorAdd( self.args + other.args )
            else:
                return VectorAdd( self.args + (other,) )
        else:  ## i.e. self is not a VectorAdd object
            if type(other)==VectorAdd:
                return VectorAdd( (self,) + other.args )
            else:
                return VectorAdd( (self , other) )

    ## Scalar multiplication for vector
    def scal_vec_mul(self, scalar):
        if type(self)==VectorMul:
            return VectorMul( scalar * self.coeffi , self.vector )
        elif type(self)==VectorAdd:
            return self.distribution_law(scalar)
        else:
            return VectorMul( scalar, self)

    ## Method to get 'free variables' for vector as set
    def get_inde_vec(self):
        return { self }

    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        return set()

    ## Alternative for ==, avoiding hashable issue
    def equals(self,other):
        return self - other == ZeroVector

    ## For Pretty printing,
    def check_minus_sign_in_coeffi(self):
        if is_number(self.coeffi):
            if self.coeffi<0:
                return True
            else:
                return False
        else:
            if self.coeffi.num_coeffi < 0:
                return True
            else:
                return False

In [262]:
## ZeroVector object : to define method equals for vectors
class ZeroVectorClass(Vector):
    def __str__(self):
        return "O"

    def __repr__(self):
        return "O"

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)

    def __hash__(self):
        return hash("ZeroVector")

    def __neg__(self):
        return self

ZeroVector = ZeroVectorClass("Zerovector")

In [263]:
## Class scalar multiple of vector
class VectorMul(Vector):
    def __init__(self, coeffi, vector):
        self.coeffi = coeffi
        self.vector = vector

    def __new__(cls, coeffi, vector):
        if coeffi==0:
            return ZeroVector
        if coeffi==1:
            return vector
        return super().__new__(cls)

    def __str__(self):
        s = ""
        if type(self.coeffi) == ScalarAdd:
            s += "( " + str(self.coeffi) + " )"
            s += MulScalVecStr
        elif self.coeffi==1:
            s += ""
        elif self.coeffi==-1:
            s += "-"
        else:
            s += str(self.coeffi)
            s += MulScalVecStr
        if type(self.vector) == VectorAdd:
            return s + "( " + str(self.vector) + " )"
        else:
            return s + str(self.vector)

    ## Method to get 'free variables' for vector as set
    def get_inde_vec(self):
        return { self.vector }

    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        return self.vector.get_inde_Point()

    ## ex) (a+b)^2 * v = (a^2 + 2ab + b^2) * v
    def expand(self):
        try:
            return self.coeffi.expand() * self.vector
        except:
            return self

    def substitute(self, sub_dict):
        try:
            coeffi = self.coeffi.substitute(sub_dict)
        except:
            coeffi = self.coeffi
        vector = self.vector.substitute(sub_dict)
        return coeffi * vector

In [264]:
class VectorAdd(Vector):
    def __init__(self, args):
        self.args = self.organize_args(args)

    def __new__(cls, args):
        organized_args = cls.organize_args(cls, args)
        if organized_args == ():
            return ZeroVector
        elif len(organized_args) == 1:
            return organized_args[0]
        return super().__new__(cls)

    def __str__(self):
        s = ""
        s += str(self.args[0])
        for i in range( 1, len(self.args) ):
            if self.args[i].check_minus_sign_in_coeffi():
                s += str( self.args[i] )
            else:
                s += " + " + str( self.args[i] )
        return s

    ## For VectorAdd,
    def distribution_law(self, scalar):
        result_args = ()
        for arg in self.args:
            arg_to_add = arg.scal_vec_mul(scalar)
            if arg_to_add != ZeroVector:
                result_args += (arg_to_add,)
        return VectorAdd(result_args)

    ## Method to get 'free variables' for vector as set
    def get_inde_vec(self):
        inde_vecs = set()
        for arg in self.args:
            inde_vecs.add( arg.vector )
        return inde_vecs

    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        inde_Points = set()
        for arg in self.args:
            inde_Points = inde_Points.union( arg.vector.get_inde_Point() )
        return inde_Points

    ## Organize linear combination with indepedent variables
    def organize_args(self, args):
        og = ArgsOrganizer()
        return og.organize_VectorAdd_args(args)

    ## expansion for scalar coefficient : ex) (a+b)^2 * v + w = (a^2 + 2ab + b^2) * v + w
    def expand(self):
        result = ZeroVector
        for arg in self.args:
            result = result + arg.expand()
        return result

    def substitute(self, sub_dict):
        result = ZeroVector
        for arg in self.args:
            result += arg.substitute(sub_dict)
        return result

---
# 'Subclasses'

## 'Subclasses' of  Scalars

### Inner Product

In [265]:
## Inner Product
class IP(Scalar):
    is_IP = True

    def __init__(self, args0, args1):
        if ( not is_vector(args0) ) or ( not is_vector(args1) ):
            raise TypeError( "Can't innerproduct" + str(type(args0)) + " with " + str(type(args1)) )
        og = ArgsOrganizer()
        self.args = og.sort_args( (args0, args1) )
        self.IP_term = self
        self.base = self

    ## change IP expression to <,> and ||.||^2
    def __str__(self):
        if self.args[0]==self.args[1]:
            s = "|| " + str(self.args[0]) + " ||^2"
        else:
            s = "< "+ str(self.args[0]) + ', ' + str(self.args[1]) + " >"
        return s

    ## change IP expression to <,> and ||.||^2
    def __repr__(self):
        if self.args[0]==self.args[1]:
            s = "|| " + repr(self.args[0]) + " ||^2"
        else:
            s = "\langle "+ repr(self.args[0]) + ', ' + repr(self.args[1]) + "\\rangle"
        return s

    ## Get 'free variable' of IP terms
    def get_inde_IP(self):
        return { self }

    ## Get 'free variable' of FV terms
    def get_inde_FV(self):
        return set()

    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        result = set()
        result = result.union( self.args[0].get_inde_Point() )
        result = result.union( self.args[1].get_inde_Point() )
        return result

    def expand(self):
        result = 0
        if type(self.args[0])==VectorAdd and type(self.args[1])==VectorAdd:
            for arg_0 in self.args[0].args:
                for arg_1 in self.args[1].args:
                    result += IP(arg_0, arg_1)
        elif type(self.args[0])==VectorAdd:
            for arg_0 in self.args[0].args:
                result += IP(arg_0, self.args[1])
        elif type(self.args[1])==VectorAdd:
            for arg_1 in self.args[1].args:
                result = IP(self.args[0], arg_1)
        else:
            return self.args[0].coeffi * self.args[1].coeffi * IP(self.args[0].vector, self.args[1].vector)
        return result

    def substitute(self, sub_dict):
        arg0 = self.args[0].substitute(sub_dict)
        arg1 = self.args[1].substitute(sub_dict)
        return IP(arg0, arg1)

# square
class Square(IP):
    def __init__(self, arg):
        super().__init__(arg, arg)

### Function Value

In [266]:
## Fucntion Vlaue
class FV(Scalar):
    is_FV = True

    def __init__(self, name, x):
        super().__init__(name)
        self.input = x
        self.FV_term = self

    def __str__(self):
        s = self.name + "(" + str(self.input) + ")"
        return s

    ## Get 'free variable' of IP terms
    def get_inde_IP(self):
        return set()

    ## Get 'free variable' of FV terms
    def get_inde_FV(self):
        return { self }

    def substitute(self, sub_dict):
        new_input = self.input.substitute(sub_dict)
        if self.input!=new_input:
            return FV(self.name, new_input)
        else:
            return super().substitute(sub_dict)

## class inherits Function to produce FV object
class Function():
    def __init__(self,name):
        self.name = name

    def __call__(self,x):
#         name = self.name + "(" + str(x) + ")"
        return FV(self.name, x)

#### IterationCount

In [267]:
## class for object about number of iteration, iteration may start with 0 or 1
## keyword argument assumptions are not yet implemented
class IterationCount(Scalar):
    is_integer = True

    def __init__(self, name, **kwargs):
        if 'start' in kwargs:
            if kwargs['start'] == 0:
                return Scalar.__init__(self, name, integer=True, negative=False, **kwargs)
            if kwargs['start'] == 1:
                return Scalar.__init__(self, name, integer=True, positive=True, **kwargs)
        else:
            return Scalar.__init__(self, name, interger=True, positive=True, **kwargs)

## 'Subclasses' of  Vector

### Classes to represent main terms in algorithm

In [268]:
## class for vectors such as x^k, y^k, nable.f(x)
class Point(Vector):
    ## Method to get 'free variables' for Point as set
    def get_inde_Point(self):
        return { self }

In [269]:
class OV(Vector):
    is_OV = True

    def __init__(self, name, x):
        super().__init__(name)
        self.input = x
        self.OV_term = self

    def __str__(self):
        s = self.name + "(" + str(self.input) + ")"
        return s

    ## Get 'free variable' of IP terms
    def get_inde_IP(self):
        return set()

    ## Get 'free variable' of FV terms
    def get_inde_OV(self):
        return { self }

    def substitute(self, sub_dict):
        new_input = self.input.substitute(sub_dict)
        if self.input!=new_input:
            return OV(self.name, new_input)
        else:
            return super().substitute(sub_dict)


## class for Operators such as gradient
class Operator:
    def __init__(self,name):
        self.name = name
        self.zeros = []

    def __call__(self,x:Vector):
#         name = self.name + "(" + str(x) + ")"
#         return Point(name)
        return OV(self.name, x)

    def zero(self,name):
        v = Point(name)
        self.zeros.append(v)
        return v

In [270]:
g = Operator('g')
print(-g(x(k)))

a=-1
print(a*g(x(k)))
print(3*g(x(k)))

-g(x^{k})
-g(x^{k})
3 * g(x^{k})


In [271]:
## class to express algorithm settings with parameter k
class Sequence:
    def __init__(self,name):
        self.name = name
        self.dict = {} # XXX 임시방편

    def __call__(self, k):
        if k in self.dict:
            return self.dict[k]
        else:
            name = self.name + "^{" + str(k) + "}"
            return Vector(name)

    # XXX 임시방편
    def add_dict(self, index, value):
        self.dict[index] = value

In [272]:
class ScalarSequence:
    def __init__(self,name):
        self.name = name
        self.dict = {} # XXX 임시방편

    def __call__(self, k):
        # XXX 임시방편
        if k in self.dict:
            return self.dict[k]
        else:
            name = self.name + "^{" + str(k) + "}"
        return Scalar(name)

    # XXX 임시방편
    def add_dict(self, index, value):
        self.dict[index] = value

### Equality

In [273]:
#Equality
class Equal():
    def __init__(self, lhs, rhs):
        self.args = (lhs, rhs)
        self.istrue = 2
        self.lhs = lhs
        self.rhs = rhs

    def __str__(self):
        s = str(self.args[0]) + " = " + str(self.args[1])
        return s

    def __repr__(self):
        s = repr(self.args[0]) + " = " + repr(self.args[1])
        return s


    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)

    def __eq__(self, other):
        if isinstance(other, Equal):
            return self.lhs == other.lhs and self.rhs == other.rhs

    def __hash__(self):
        return hash((self.lhs, self.rhs))


    def substitute(self, sub_dict):
        # self.args = (self.args[0].substitute(sub_dict), self.args[1].substitute(sub_dict) )
        return Equal(self.args[0].substitute(sub_dict), self.args[1].substitute(sub_dict) )


    def eq_trans(self,eq2):
        if self.args[1] == eq2.args[0]:
          eq = Equal(self.args[0],eq2.args[1])
          eq.istrue = 1
          return eq
        else:
          raise TypeError( "Can't apply transitivity.")

    def eq_sym(self):
        eq = Equal(self.args[1], self.args[0])
        eq.istrue = 1
        return eq


    def typecheck(self):
        if type(self.lhs) is type(self.rhs):
            return type(self.lhs)
        else: raise TypeError( "LHS and RHS have different types" )



#Var is a set of variables used. For later use
Var = set()

# Prop constitutes of pairs of proposition and its case(proved,unproved,axiom)
# propositions can be Ineq / Equality / function (forall case).
Prop = set()

Tactic = set()

---
# Example1 : Gradient descent
$$ x^{k+1} = x^k - \frac{1}{L} \nabla f(x^k) $$
---

## Define materials

In [274]:
x = Sequence('x') # seqeunce generated by gradient descent
f = Function('f')
g = Operator('g') # Grdaient f
k = IterationCount('k')
L = Scalar('L')   # smoothness parameter
A = ScalarSequence('A')

In [275]:
x_star = x('\star')
f_star = f(x_star)

### Define Lyapunov function
$$ \Phi^n = A^n ( f(x^n) - f^\star ) + \frac{L}{2} \left\| x^n - x^\star \right\|^2 $$

In [276]:
def Lypunov(n):
    return A(n) * ( f(x(n)) - f_star )+ L/2 * Square( x(n)-x_star )

## The 'proof' we want to show is
\begin{align*}
    \Phi^{k+1} - \Phi^{k} = \lambda_1 \left( f(x^k) - f^\star + \left\langle \nabla f(x^k) , x^\star x^k \right \rangle \right) + \lambda_2 \left( f(x^{k+1}) - f(x^k) + \left\langle \nabla f (x^k) , x^{k+1} - x^k \right\rangle + \frac{L}{2} \left\| x^k - x^{k+1} \right\|^2  \right) - \frac{1}{2L} \left\| \nabla f (x^k) \right\|^2
\end{align*}
### where
\begin{align*}
    \lambda_1 &= A^{k+1} - A^k,  \qquad
    \lambda_2 = A^{k+1}
\end{align*}

In [277]:
lambda_1 = A(k+1) - A(k)
lambda_2 = A(k+1)

In [278]:
ineq1 = f(x(k)) - f_star + IP(g(x(k)), x_star-x(k))
ineq2 = f(x(k+1)) - (f(x(k)) + IP(g(x(k)),x(k+1)-x(k)) + L/2 * Square(x(k)-x(k+1))  )

In [279]:
LHS = Lypunov(k+1) - Lypunov(k)
RHS = lambda_1 * ineq1 + lambda_2 * ineq2 - 0.5/L * A(k) * Square(g(x(k)))

In [280]:
( LHS - RHS ).simplify()

( -0.5 L+0.5 A^{1+k} L ) || x^{k} ||^2+( -A^{1+k}+A^{k} ) < g(x^{k}), x^{\star} >+( 0.5 A^{1+k} L+0.5 L ) || x^{1+k} ||^2-A^{1+k} L < x^{1+k}, x^{k} >-A^{k} < g(x^{k}), x^{k} >-L < x^{1+k}, x^{\star} >+0.5 A^{k} L^{-1} || g(x^{k}) ||^2+A^{1+k} < g(x^{k}), x^{1+k} >+L < x^{\star}, x^{k} >

## Substituting Example

In [281]:

def x_def(n) :
  statement = Equal(x(n+1), x(n)-1/L*g(x(n)))
  statement.truth = 'Axiom'
  return statement

def A_def(n):
  statement = Equal(A(n+1), A(n)+1)
  statement.truth = 'Axiom'
  return statement


# def A_subs(n):
#     true_eq = {}
#     if n.is_integer:
#         true_eq[A(n+1)] = A(n) + 1
#         return true_eq


# def GD(n):
#     true_eq = {}
#     if n.is_integer:
#         true_eq[x(n+1)] = x(n) - 1/L * g(x(n))
#         return true_eq

In [282]:
( LHS - RHS ).simplify()

( -0.5 L+0.5 A^{1+k} L ) || x^{k} ||^2+( -A^{1+k}+A^{k} ) < g(x^{k}), x^{\star} >+( 0.5 A^{1+k} L+0.5 L ) || x^{1+k} ||^2-A^{1+k} L < x^{1+k}, x^{k} >-A^{k} < g(x^{k}), x^{k} >-L < x^{1+k}, x^{\star} >+0.5 A^{k} L^{-1} || g(x^{k}) ||^2+A^{1+k} < g(x^{k}), x^{1+k} >+L < x^{\star}, x^{k} >

In [283]:
( LHS - RHS ).substitute(x_def(k)).simplify()

( -0.5 A^{1+k} L^{-1}+0.5 A^{k} L^{-1}+0.5 L^{-1} ) || g(x^{k}) ||^2+( -1.0-A^{k}+A^{1+k} ) < g(x^{k}), x^{k} >+( 1-A^{1+k}+A^{k} ) < g(x^{k}), x^{\star} >

In [284]:
( LHS - RHS ).substitute(x_def(k)).substitute(A_def(k)).simplify()

0

### Equality and Basic Settings

In [285]:
### Example axioms ###

def eq_to_le(eq) :  #equal implies <=
  p = Ineq(eq.args[0],eq.args[1])
  p.istrue = eq.istrue
  return p


def eq_refl(a):
  eq = Equal(a,a)
  eq.istrue = 1
  return eq

def eq_trans(eq1,eq2):
  if eq1.args[1] == eq2.args[0]:
    eq = Equal(eq1.args[0],eq2.args[1])
    eq.istrue = eq1.istrue * eq2.istrue
    return eq
  else:
    raise TypeError( "Can't apply transivity.")

def eq_sym(eq):
  result = Equal(eq.args[1],eq.args[0])
  result.istrue = eq.istrue
  return result




In [286]:
# Example
print(eq_refl(L))
print(eq_to_le(eq_refl(L)))
print(eq_refl(x(k+1)).substitute((x_def(k))))

L = L
L <= L
-L**{-1} * g(x^{k}) + x^{k} = -L**{-1} * g(x^{k}) + x^{k}


## Manage Proposition(Assumption) Data & Verify

In [287]:
def verify_equality(eq,proof):
  #Look up in Prop
  if eq in Prop:
      return True

  if isinstance(eq,neg):
      return not verify_equality(eq.inside, proof)

  #For now, we only consider substitute and simplify in proving steps
  #Using each steps of proof : ordered list of statements in Prop
  result = eq.lhs-eq.rhs
  if is_number(result):
      return result == 0

  for step in proof:
      if step in Prop:
        if isinstance(step, Equal):                 #substitution
          result = result.substitute(step)
        elif isinstance(step, lemma):               #add lemma
          propadd_eq(lemma.claim, lemma.proof)
        elif isinstance(step, tactic):
          #if step.tactic in Tactic:
          if callable(step.tactic):               #tactic
              step.tactic(step.claim)
        else: raise TypeError(f'Cannot use the following step: {step}' )
      else:
        raise TypeError( f'Should first verify the current step: {step}')

  #Check if LHS-RHS = 0
  if result == 0:
      return True
  elif result.simplify() == 0:
      return True
  else:
      return result.simplify().is_zero()


class neg():
    def __init__(self, prop):
        self.inside = prop
        self.is_equality = 0
        self.is_ineq = 0
        self.istrue = 2 if prop.istrue == 2 else 1 - prop.istrue

    def __str__(self):
        s = f'not_( {self.inside} )'
        return s

    def __repr__(self):
        s = " \\neg ( " + repr(self.inside) + ")"
        return s

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)



def propadd_assumption(prop):
  #if prop.istrue == 'assumed' :
  Prop.add(prop)


def propadd_eq(eq,proof):
  if verify_equality(eq,proof) == True:
    Prop.add(eq)
    #print(f'{eq} added to Proposition data')
  else: print('failed to verify')



us = Scalar('us') #universal scalar
uv = Vector('uv') #universal vector

def propadd_forall_scalar(forall_statement,proof): #For now this works for equalities with only one forall quantifier
  u = us
  if verify_equality(forall_statement(u),proof = []) == True:
    Prop.add (forall_statement)
    print('added to Proposition data')
  else: print('Failed to verify')

def propadd_forall_vector(forall_statement,proof): #For now this works for equalities with only one forall quantifier
  u = uv
  if verify_equality(forall_statement(u),proof = []) == True:
    Prop.add (forall_statement)
    print(f'{forall_statement} added to Proposition data')
  else: print('Failed to verify')

def propadd_intro(fun,u):
    if fun in Prop:
      Prop.add(fun(u))
    else: print('Failed to verify')





In [288]:
Prop = set()
known_truth = {eq_refl,eq_sym,eq_to_le,eq_trans}




propadd_assumption(x_def) #definition of x
propadd_assumption(A_def)
# we will only allow adding props constructed from Prop data.
propadd_intro(x_def,k)
propadd_intro(A_def,k)



Prop = Prop.union(known_truth)
Prop0 = Prop.copy()
print(Prop)





{x^{1+k} = -L^{-1} g(x^{k}) + x^{k}, <function eq_refl at 0x000001EC78C4F040>, A^{1+k} = 1+A^{k}, <function eq_trans at 0x000001EC78C4F0D0>, <function x_def at 0x000001EC78C18EE0>, <function eq_to_le at 0x000001EC77C0AEE0>, <function eq_sym at 0x000001EC78C4F160>, <function A_def at 0x000001EC77BFF9D0>}


In [289]:
#example 1 : grad descent for fixed k

exmpl_1 = Equal(LHS,RHS)
print(verify_equality(exmpl_1, [x_def(k),A_def(k)]))
propadd_eq(exmpl_1, [x_def(k),A_def(k)])
print(exmpl_1 in Prop)



#example 2 : adding zero

def add_zero_vec(x):
  eq = Equal(x, x + ZeroVector)
  eq.istrue = 'true'
  return eq

propadd_forall_vector(add_zero_vec,[])
print(add_zero_vec in Prop)


True
True
<function add_zero_vec at 0x000001EC77B52550> added to Proposition data
True


### Example: constructing 2-Step of GD :
$$x^{k+1} = x^{k-1} -\frac1L(\nabla f(x^k) + \nabla f(x^{k-1})) $$

In [290]:
x = Sequence('x') # seqeunce generated by gradient descent
f = Function('f')
g = Operator('g') # Grdaient f
k = IterationCount('k')
L = Scalar('L')   # smoothness parameter
A = ScalarSequence('A')



def x_def(n) :
  statement = Equal(x(n+1), x(n)-1/L*g(x(n)))
  statement.istrue =  'def'
  return statement

def A_def(n):
  statement = Equal(A(n+1), A(n)+1)
  statement.istrue = 'def'
  return statement

In [291]:
Double_step =  Equal( x(k+1) , x(k-1) - 1/L*(g(x(k))+g(x(k-1))))
print(f'double step GD = {Double_step}')


Prop= set()
#verify_equality(Double_step,[x_def(k),x_def(k-1)])

Prop.add(x_def(k))
Prop.add(x_def(k-1))
print(Prop)
verify_equality(Double_step,[x_def(k),x_def(k-1)])



double step GD = x^{1+k} = -L**{-1} * g(x^{-1+k})-L**{-1} * g(x^{k}) + x^{-1+k}
{x^{1+k} = -L^{-1} g(x^{k}) + x^{k}, x^{k} = -L^{-1} g(x^{-1+k}) + x^{-1+k}}


True

In [292]:
Prop0 = Prop

Possible mechanism for existence

In [293]:
class exists():
  def __init__(self,eq_fun):
      self.predicate = eq_fun
      self.istrue = 'unknown'

  def case(self, var):
      Var.add(var)
      Prop.add(self.predicate(var))

  def __hash__(self):
    return hash(self.predicate)

  def __eq__(self, other):
    if isinstance(other, exists):
      return self.predicate == other.predicate
    return False

def verify_existence(ex,var,proof):  #works for one variable
  if verify_equality(ex.predicate(var),proof) == True:
      #propadd_eq(ex.predicate(var),proof)
      Prop.add(ex)

In [294]:
#example
def exmpl_3(n):
  return Equal(x(n),x(k)-1/L*g(x(k)))

verify_existence(exists(exmpl_3),(k+1),[x_def(k)])

exists(exmpl_3) in Prop


True

In [295]:
# def ifthen(p1, p2): #for now we only care about equality
#   if verify_equality(p1) == True:
#     if

Ineq

In [296]:
# class Le():

# class Lt():

# class Ge():

# class Gt():


# def le_to_ge(le):
#   return Ge(le.rhs, le.lhs)

# def square_nonneg(a):
#   return Le(0,a*a)

# def sign(const):
#   if const < 0:
#     return Lt(const,0)
#   elif const == 0:
#     return Equal(const,0)
#   else:
#     return Lt(0,const)


# n>0 => x(n+1) = x(n) ~

# 0624
### From here, we focus on AGM. First, we reset Prop/Var and define basic objects.

In [297]:
known_truth = {eq_refl,eq_sym,eq_to_le,eq_trans}
Prop = known_truth.copy()
Var = set()


#Basic Setting
t = ScalarSequence('t') #theta

x = Sequence('x')
y = Sequence('y')


X = Sequence('X') #For equivalent formulation
Y = Sequence('Y')
Z = Sequence('Z')


f = Function('f')
g = Operator('g') # Grdaient f
k = IterationCount('k')
L = Scalar('L')   # L-smoothness

## AGM has two equivalent definition of x(k), y(k). We should first define equivalence or equality of sequence.

In [298]:
# First try: just understand as forall k, x(k) = X(k) and enable forall equality
class fun_equal():
    def __init__(self, lhs, rhs):
        self.args = (lhs, rhs)
        self.istrue = 'unknown'
        self.lhs = lhs
        self.rhs = rhs

    def __str__(self):
        s = str(self.args[0]) + " = " + str(self.args[1])
        return s

    def __repr__(self):
        s = repr(self.args[0]) + " = " + repr(self.args[1])
        return s


    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)

    def __eq__(self, other):
        if isinstance(other, fun_equal):
            return self.lhs == other.lhs and self.rhs == other.rhs

    def __hash__(self):
        return hash((self.lhs, self.rhs))

    def case(self, n):
        Prop.add(Equal(self.lhs(n),self.rhs(n)))
        return Equal(self.lhs(n),self.rhs(n))

    def typecheck(self):
        if type(self.lhs) is type(self.rhs):
            return type(self.lhs)
        else: raise TypeError( "LHS and RHS have different types" )
#Also works for sequence
#After showing equivalence, we can use case(n) to substitute anytime.

## Verifying Equality by Induction

In [299]:
def induction_eq(forall_eq,proof1, proof2):
  case_0 = verify_equality(forall_eq(0), proof1)

  induction_hypothesis = forall_eq(us)
  Prop.add(induction_hypothesis) #us is a universal( fake ) variable so in no case this is in Prop or used later.
  case_succ = verify_equality(forall_eq(us+1), proof2 )
  Prop.remove(induction_hypothesis)

  if case_0 == True and case_succ == True:
    Prop.add(forall_eq)
    print('Induction step verified and added to Prop.')
  else:
    raise ValueError("Should verify induction step")


def induction_eq_ver2(fun_equal,proof1, proof2):
  lhs, rhs = fun_equal.lhs, fun_equal.rhs
  case_0 = verify_equality(Equal(lhs(0), rhs(0)), proof1)


  induction_hypothesis = Equal(lhs(us), rhs(us))
  Prop.add(induction_hypothesis) #us is a universal( fake ) variable so in no case this is in Prop or used later.
  case_succ = verify_equality(Equal(lhs(us+1), rhs(us+1)), proof2)
  Prop.remove(induction_hypothesis)

  if case_0 == True and case_succ == True:
    Prop.add(fun_equal)
    print('Induction step verified and added to Prop.')
  else:
    raise ValueError("Should verify induction step")


In [300]:
#induction toy example 1
def toy(n):
  return Equal((n+1)*(n+1)*(n+1), n*n*n + 3*n*n + 3*n + 1)

induction_eq(toy,[],[])




#induction toy example 2
toy_fun = Function('toy_fun')

toy_fun_equal = fun_equal(toy_fun, toy_fun)
induction_eq_ver2(toy_fun_equal, [],[])

Prop = known_truth.copy()

Induction step verified and added to Prop.
Induction step verified and added to Prop.


## Unifinished

In [301]:
# testeq = fun_equal(x, X)
# form1 = Equal(y(k+1),x(k)-1/L*g(x(k)))



# print(testeq.case(k))

# print(form1)

# print(form1.substitute(testeq.case(k))) #ㅇㄴ 왜 치환 안댐

# print(testeq.case(k) == Equal(x(k),X(k)))




In [302]:
# class VFV(Vector): #No innerproduct or indep defined.
#     is_VFV = True

#     def __init__(self, name, x):
#         super().__init__(name)
#         self.input = x
#         self.VFV_term = self

#     def __str__(self):
#         s = self.name + "(" + str(self.input) + ")"
#         return s

#     # def __mul__(self, other):
#     #     if isinstance(other, Scalar):  # Assuming Scalar is a superclass for scalar objects

#     #         return VectorMul(other, self)
#     #     else:
#     #         print('bbb')
#     #     # handle other types of multiplication if necessary

#     # def __rmul__(self, other):
#     #      if isinstance(other, Scalar):  # Assuming Scalar is a superclass for scalar objects
#     #         print(f'aaaa')
#     #         return VectorMul(other, self)
#     #      else:
#     #         print('bbb')
#     def substitute(self, sub_dict):
#         new_input = self.input.substitute(sub_dict)
#         if self.input!=new_input:
#             return VFV(self.name, new_input)
#         else:
#             return super().substitute(sub_dict)


# ## class for Operators such as gradient
# class vec_valued_fun:
#     def __init__(self,name):
#         self.name = name
#         self.zeros = []


#     def __call__(self,x):
#         return VFV(self.name, x)

#     def zero(self,name):
#         v = Vector(name)
#         self.zeros.append(v)
#         return v

In [303]:
# grad_f = vec_valued_fun('grad_f')


# #form1 = Equal(y(k+1),x(k)-1/L*grad_f(x(k)))

In [304]:
# type(L*grad_f(x(k)))

In [305]:
# print(type(grad_f(x(k))))
# print(type(L*grad_f(x(k))))

In [306]:
# print(type(f(x(k))))
# print(type(L*f(x(k))))


## AGM x,y equivalence

In [307]:
#Fomulation 1

def t_def(n):
  return Equal( t(n+1)*t(n+1) - t(n+1), t(n)*t(n) )

assumption1 = Equal(t(0),1)
assumption2 = Equal(y(0),x(0))

def form1eq1(n):
  return Equal(y(n+1), x(n)-1/L*g(x(n)))

def form1eq2(n):
  return Equal(x(n+1) , y(n+1) + (t(n)-1)/t(n+1)*(y(n+1)-y(n)))


Prop.update([t_def,assumption1,assumption2,form1eq1,form1eq2])
print(Prop)


{y^{0} = x^{0}, <function eq_refl at 0x000001EC78C4F040>, t^{0} = 1, <function eq_trans at 0x000001EC78C4F0D0>, <function eq_to_le at 0x000001EC77C0AEE0>, <function form1eq1 at 0x000001EC77A3FAF0>, <function form1eq2 at 0x000001EC77B52700>, <function eq_sym at 0x000001EC78C4F160>, <function t_def at 0x000001EC78C4FB80>}


In [308]:
#Formulation 2
assumption3 = Equal(Y(0) , X(0))
assumption4 = Equal(Z(0) , X(0))
assumption5 = Equal(x(0) , X(0))
assumption6 = Equal(y(0) , Y(0))


def form2eq1(n):
  return Equal(Y(n+1), X(n) - 1/L*g(x(n)))

def form2eq2(n):
  return Equal(Z(n+1), Z(n) - t(n)/L*g(x(n)))

def form2eq3(n):
  return Equal(X(n+1), (1- 1/t(n+1))*Y(n+1) + 1/t(n+1)*Z(n+1) )



Prop.update([assumption3, assumption4, assumption5, assumption6, form2eq1, form2eq2, form2eq3])
print(Prop)

{y^{0} = x^{0}, x^{0} = X^{0}, <function eq_refl at 0x000001EC78C4F040>, Z^{0} = X^{0}, t^{0} = 1, <function form2eq1 at 0x000001EC77A40280>, Y^{0} = X^{0}, <function eq_trans at 0x000001EC78C4F0D0>, <function eq_to_le at 0x000001EC77C0AEE0>, <function form1eq1 at 0x000001EC77A3FAF0>, <function form1eq2 at 0x000001EC77B52700>, <function form2eq2 at 0x000001EC77A82B80>, <function eq_sym at 0x000001EC78C4F160>, <function t_def at 0x000001EC78C4FB80>, <function form2eq3 at 0x000001EC77A82DC0>, y^{0} = Y^{0}}


In [309]:
y_equiv = fun_equal(y,Y)


#preparation
propadd_intro(form1eq1,us) #y(us+1) = x(us)-1/L*g(x(us))
propadd_intro(form2eq1, us)
Prop.add(Equal(x(us),X(us))) # Temp


#Generating Proof
proof1 = [assumption6] # y(0) = Y(0)

proof2 = [ ]
proof2.append( form1eq1(us) ) #y(us+1) = x(us)-1/L*g
proof2.append(Equal(x(us),X(us)) ) # rewrite as: = X(us)-1/L*g

hY = form2eq1(us)  #X(us) - 1/L*g = Y(us+1)
proof2.append(hY)


induction_eq_ver2(y_equiv, proof1, proof2)



ValueError: Should verify induction step

In [310]:
tmp_0 =  1/t(k)
tmp_1 = 1- tmp_0
tmp_2 = (t(k)-1)/t(k)

tmp = Equal(1-1/t(k),(t(k)-1)/t(k))
verify_equality(tmp,[])

True

# **0703**

Goal: positivity of θ, division by nonzero

First, we implement inequality (for scalars)

In [311]:
class le():
    def __init__(self, lhs, rhs):
        self.args = (lhs, rhs)
        self.lhs = lhs
        self.rhs = rhs
        self.istrue = 2. # 0=false / 1=true / 2=unknown

    def __str__(self):
        s = str(self.args[0]) + " <= " + str(self.args[1])
        return s

    def __repr__(self):
        s = repr(self.args[0]) + " \\le " + repr(self.args[1])
        return s

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)

    def __eq__(self, other):
        if isinstance(other, le):
            return self.lhs == other.lhs and self.rhs == other.rhs

    def __hash__(self):
        return hash((self.lhs, self.rhs))


class lt():
    def __init__(self, lhs, rhs):
        self.args = (lhs, rhs)
        self.lhs, self.rhs = lhs, rhs
        self.istrue = 2

    def __str__(self):
        s = str(self.args[0]) + " < " + str(self.args[1])
        return s

    def __repr__(self):
        s = repr(self.args[0]) + " \\lt " + repr(self.args[1])
        return s

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)

    def __eq__(self, other):
        if isinstance(other, le):
            return self.lhs == other.lhs and self.rhs == other.rhs

    def __hash__(self):
        return hash((self.lhs, self.rhs))



#### for logical expression (temporary)







class ifthen():
    def __init__(self, prop1, prop2):
        self.condition = prop1
        self.consequence = prop2
        self.istrue = 2

        if prop1.istrue == 0 or prop2.istrue == 1:
          self.istrue = 1



    def __str__(self):
        s = str(self.condition) + " => " + str(self.consequence)
        return s

    def __repr__(self):
        s = repr(self.condition) + " \\implies " + repr(self.condition)
        return s

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)


class  prop_and():
    def __init__(self, prop_list):
        self.prop_list = prop_list
        self.istrue = 1
        for prop in prop_list:
                self.istrue *= prop in Prop # or prop.istrue


    def __str__(self):
        s = ""
        for prop in self.prop_list:
            s+=f'{prop} & '
        return s[:-3]


    def __repr__(self):
        s = ""
        for prop in self.prop_list:
            s+=f'{prop} \\And '
        return s[:-3]

    def _repr_latex_(self):
        return "$\\displaystyle %s$" % repr(self)

    def proj(self,n):
        result = self.prop_list[n]
        if self in Prop:
            Prop.add(result)
        return result




class lemma():
    def __init__(self, claim, proof):
        self.claim = claim
        self.proof = proof
        self.istrue = 2


class tactic():
    def __init__(self, tactic, claim):
        self.claim = claim
        self.tactic = tactic
        #self.arguments = arguments

    def specify(self,arguments):
        if callable(self.arguments):
          return tactic(self.tactic, self.claim(arguments))
        else: raise ValueError('Check the claim type')





## Basic properties

In [65]:


#double negation
def add_to_Prop_if_double_neg_in_Prop(prop):
    if isinstance(prop, neg) and isinstance(prop.inside, neg):
        prop.istrue = prop.inside.inside.istrue
        if prop in Prop:
            Prop.add(prop.inside.inside)



# =>
def MP(ifthen, P):
    if ifthen.condition == P:
        result = ifthen.consequence

        if (ifthen.istrue ==1 or ifthen in Prop) and (P.istrue ==1 or P in Prop):
            result.istrue = 1
            Prop.add(result)
        return result

    else: raise TypeError('Wrong Input for Modus Ponens')


# a less than b implies a less than or equal to b
def lt_to_le(lt):
    result = le(lt.lhs, lt.rhs)
    if lt.istrue == 1:
       result.istrue = 1
       Prop.add(result)
    return result

# if a<b the a \neq b.
def lt_to_neq(lt):
    result = neg(Equal(lt.lhs,lt.rhs))
    if lt.istrue ==1:
        result.istrue = 1  #even if lt.istrue == false, this can be true.
        Prop.add(result)
    return result

#if x <= y but x!=y, then x<y.
def le_and_neq_to_lt(le,neq):
    if le.lhs == neq.inside.lhs and le.rhs == neq.inside.rhs:
        result = lt(le.lhs, le.rhs)
        if le.istrue == 1 and neq.istrue == 1:
            result.istrue = 1
            Prop.add(result)
    return result


#nonegativity of square
def square_nonneg(scalar):
    result = le(0 , scalar * scalar)
    result.istrue = 1
    Prop.add(result)
    return result

#sum of nonneg is nonneg


#trichotomy

#python numbers
def le_of_numbers(a,b):
  if is_number(a) and is_number(b):
    if a<=b:
      result = le(a,b)
      result.istrue = 1
    else: result = lt(b,a)
    return result
  else: raise TypeError( "Please check the input types")


def lt_of_numbers(a,b):
  if is_number(a) and is_number(b):
    if a<b:
      result = lt(a,b)
      result.istrue = 1
    else: result = le(b,a)
    return result
  else: raise TypeError( "Please check the input types")


## Division and multiplication: should preserve truth value, and should not divide by zero
 & addition

In [66]:
def divide_equality_by_nonzero(eq, divisor):
    if neg(Equal(divisor,0)) in Prop or neg(Equal(0,divisor)) in Prop and isinstance(eq,Equal) :  # divisor != 0 & eq : Equal
        result = Equal( eq.lhs / divisor , eq.rhs / divisor )
        if eq.istrue == 1 or eq in Prop:              #preserve the truth value
            result.istrue = 1
            Prop.add(result)
        return result

    else: raise ValueError( "You should verify that you are dividing an equality by nonzero.")


def divide_inequality(ineq, divisor):
    if isinstance(ineq,(le,lt)):
        if lt(0,divisor) in Prop:        #positive multiplication prserves order
            result = type(ineq)(ineq.lhs / divisor, ineq.rhs / divisor )
            if ineq.istrue == 1 or ineq in Prop:
                result.istrue = 1
                Prop.add(result)
            return result

        if lt(divisor, 0) in Prop:       #negative multiplication reverses order
            result = type(ineq)(ineq.rhs / divisor, ineq.lhs / divisor )
            if ineq.istrue == 1 or ineq in Prop:
                result.istrue = 1
                Prop.add(result)
            return result

    else: raise ValueError( "You should verify that you are dividing an inequality by nonzero.")


# multiplication
def mul_inequality(ineq, multiplier):
    if isinstance(ineq,(le,lt)):
        if Equal(0,multiplier) in Prop or Equal(multiplier,0) in Prop or multiplier == 0:
            return Equal(0,0)                           #multiplication by 0

        elif le(0, multiplier) in Prop or lt(0, multiplier) in Prop or 0 < multiplier:
            result = type(ineq)(ineq.lhs * multiplier, ineq.rhs * multiplier )
            if ineq.istrue == 1 or ineq in Prop:
                result.istrue = 1
                Prop.add(result)
            return result

        elif le(multiplier, 0) in Prop or lt(multiplier, 0) in Prop or multiplier < 0:
            result = type(ineq)(ineq.rhs * multiplier, ineq.lhs * multiplier )
            if ineq.istrue == 1 or ineq in Prop:
                result.istrue = 1
                Prop.add(result)
            return result

        else: return None

    else: raise TypeError( "You should verify that you are multiplying an inequality.")


def mul_equality(eq, multiplier):
    if isinstance(eq, Equal):
        result = Equal(eq.lhs * multiplier, eq.rhs * multiplier)
        if eq.istrue == 1 or eq in Prop:
            result.istrue = 1
            Prop.add(result)

        return result
    else: raise TypeError( "You should verify that you are multiplying an equality.")

#addition
def add_of(summand, adder):
    if isinstance(summand, (Equal,le,lt)):
        result = type(summand)(summand.lhs + adder, summand.rhs + adder)
        if summand.istrue == 1 or summand in Prop:
            result.istrue = 1
            Prop.add(result)

        return result
    else: raise TypeError( "You should verify that you are adding the right type.")




In [67]:
# define le_of_lt, lt_of_le  and the new transitivity <- I think they should be able to be verified not deifned

def le_of_lt(le,lt):
    if le.rhs == lt.lhs:
       return lt(le.lhs, lt.rhs)
    else: raise TypeError('wrong input type')


def lt_of_le(lt,le):
    if lt.rhs == le.lhs:
        return lt(lt.lhs, le.rhs)
    else: raise TypeError('wrong input type')


def le_of_le(le1,le2):
    if le1.rhs == le2.lhs:
       return le(le1.lhs, le2.rhs)
    else: raise TypeError('wrong input type')


def lt_of_lt(lt1,lt2):
    if lt1.rhs == lt2.lhs:
       return lt(lt1.lhs, lt2.rhs)
    else: raise TypeError('wrong input type')


def ineq_trans(ineq1, ineq2):
    if ineq1.rhs != ineq2.lhs:
        raise TypeError('Cannot apply transitivity')


    if ineq1.ineq_type == "le" and ineq2.ineq_type == "le":
        result = le_of_le(ineq1, ineq2)

    if ineq1.ineq_type == "le" and ineq2.ineq_type == "lt":
        result = le_of_lt(ineq1, ineq2)

    if ineq1.ineq_type == "lt" and ineq2.ineq_type == "le":
        result = lt_of_le(ineq1, ineq2)

    if ineq1.ineq_type == "lt" and ineq2.ineq_type == "lt":
        result = lt_of_lt(ineq1, ineq2)

    if ineq1.istrue == 1 and ineq2.istrue == 1:
        result.istrue = 1
        Prop.add(result)

    try :  return result
    except: raise ValueError('wrong input type')



In [68]:
#PROVE: , sum_of_nonneg_is_nonneg  / sum_of_pos_is_pos


#def le_of_lt(le,lt):

1 * True * 2


2

# This part is about verifying inequality.

In [69]:
def verify_le(ineq, proof): #FIRST LET'S JUST FOCUS ON PROVING THETA

    if ineq in Prop:            #check if it is already known to be true
        return True

    lhs = ineq.lhs
    rhs = ineq.rhs

    #ineq_type = type(ineq)

    #if is_number(lhs) and is_number(rhs):

    try: return lhs <= rhs
    except:
      for step in proof:
        if step not in Prop:       #check if each proof step is verified
          raise TypeError( f'Should first verify the current step: {step}')
          return False

        if isinstance(step,Equal):
          lhs = lhs.substitute(step).simplify()
          rhs = rhs.substitute(step).simplify()

        if isinstance(step,le): #transitiivty
          if step.lhs == rhs:
            rhs = step.rhs
          if step.rhs == lhs:
            lhs = step.lhs



    return le(lhs,rhs) in Prop or 0<=rhs-lhs or 0<=(lhs-rhs).simplify() or le(lhs.simplify(),rhs.simplify()) in Prop





def verify_lt(ineq, proof): #FIRST LET'S JUST FOCUS ON PROVING THETA

  if ineq in Prop:            #check if it is already known to be true
      return True

  lhs = ineq.lhs
  rhs = ineq.rhs

  #ineq_type = type(ineq)

  #if is_number(lhs) and is_number(rhs):

  try: return lhs < rhs
  except:
    for step in proof:
      if step not in Prop:       #check if each proof step is verified
        raise TypeError( f'Should first verify the current step: {step}')
        return False

      if isinstance(step,Equal):
        lhs = lhs.substitute(step).simplify()
        rhs = rhs.substitute(step).simplify()

      if isinstance(step,lt): #transitiivty
        if step.lhs == rhs:
          rhs = step.rhs
        if step.rhs == lhs:
          lhs = step.lhs



  return lt(lhs,rhs) in Prop or 0<rhs-lhs or 0<=(lhs-rhs).simplify() or lt(lhs.simplify(),rhs.simplify()) in Prop





def propadd_ineq(ineq,proof):
    if isinstance(ineq, le):
        if verify_le(ineq,proof) == True:
           Prop.add(ineq)

        else: raise ValueError(f'{ineq} not verified')

    elif isinstance(ineq, lt):
        if verify_lt(ineq,proof) == True:
            Prop.add(ineq)

        else: raise ValueError(f'{ineq} not verified')
    else: raise TypeError(f'Wrong input prop type: {type(ineq)}.')




def propadd_ineq_intro(fun,case) :
    if fun in Prop:
      Prop.add(fun(case))




   #subsittue if a \in proof is equal, trans if a \in proof is ineq.

   #put add_ineq or mul_ineq or le_ineq



#examples
print(verify_le(le(0,1), []))
print(verify_le(le(1,0), []))
print(verify_lt(lt(0,0),[]))

True
False
False


## Sqrt

In [341]:
# class Sqrt(): #let Sqrt(Scalar)?

#   def __init__(self, inside ):
#         self.inside = inside
#         self.isreal = verify_le(le(0,self.inside),[])
#         #self.value = inside**0.5
#         #print((self,inside))
#         # if le(0, inside) not in Prop:
#         #   raise ValueError(f"You should first verify nonnegativity of {inside} inside the square root.")

#         #sqrt > 0 과 sqrt^2 = self 만 구현 아니면 모순 만들기? 모순 구현하기

#         #Prop.add(Equal(self*self, inside))

#   def __mul__(self, other):
#         if isinstance(other, Scalar):
#             return ScalarPow(self.inside,0.5) * other
#         elif isinstance(other, Sqrt):
#             #Prop.add(le(0,self.inside*other.inside))
#             return Sqrt(self.inside * other.inside)
#         else:
#             raise ValueError(f"Multiplication not supported for this type of {type(other)}")

#   def __add__(self, other):
#         if isinstance(other, Scalar) or is_number(other):
#             return ScalarPow(self.inside,0.5) + other
#         elif isinstance(other, Sqrt):
#             return ScalarPow(self.inside,0.5) + ScalarPow(other.inside,0.5)
#         else:
#             raise ValueError(f"Multiplication not supported for this type of {type(other)}")

#   def __pow__(self, exponent):  #Temp
#         if exponent == 2:
#             return self.inside
#         else: return self.inside**(exponent/2)


#   def __str__(self):
#         s = f'Sqrt({self.inside})'
#         return s

#   def __repr__(self):
#       s = " \\sqrt (" + repr(self.inside) + ")"
#       return s

#   def _repr_latex_(self):
#       return "$\\displaystyle %s$" % repr(self)


#   def __eq__(self, other):
#         if isinstance(other, Sqrt):
#             return self.inside == other.inside
#         return False

#   def __hash__(self):
#       return hash(self.inside)






# #basic properties of sqrt
# def sq_of_sqrt(a):
#     if le(0,a) in Prop:
#         result = Equal(Sqrt(a)*Sqrt(a), a)
#         Prop.add(result)
#         return result
#     else: raise ValueError('should be nonnegative inside a sqrt')


# def sqrt_nonneg(term):
#     if isinstance(term, Sqrt):
#         Prop.add(le(0,term))


In [356]:
class Sqrt(ScalarPow):
    def __init__(self, inside):
        self.base = inside
        self.exponent = 1/2

    def __new__(cls, inside):
        if type(inside)==Scalar:
            return super().__new__(cls, inside, 1/2)
        else:
            return inside**(1/2)

## Verifying for all inequality

In [346]:
def verify_forall_le(fun,proof):
  return verify_le(fun(us))



def sqrt_of_ineq(ineq):
    if isinstance(ineq,(le,lt)):
      if le(0,ineq.lhs) in Prop and le(0,ineq.rhs) in Prop:
          result = type(ineq)(ineq.lhs.sqrt(),ineq.rhs.sqrt())
          result.istrue = 1
          Prop.add(result)
          return result
    else: raise TypeError('wrong input type')


def neg_of_le(scalar):
  result = le(-scalar,0)
  if le(0,scalar) in Prop:
      result.istrue = 1
      Prop.add(result)

  return result

In [72]:


test = square_nonneg(t(k))

Prop.add(le(0,t(k)))

h_t = sq_of_sqrt(t(k))



In [73]:
verify_equality( Equal( Sqrt(t(k))**2, t(k) ), [])

True

Sum of nonneg is nonneg

In [74]:
def sum_of_nonneg(a,b):
    return ifthen( prop_and([le(0,a), le(0,b)]), le(0,a+b) )

## Ineq Induction

## Now we get back to theta

In [75]:
#Define theta
t = ScalarSequence('t')

assumption_t0 = Equal(t(0),1)
def t_def(n):
  return Equal(t(n+1), 1/2*(Sqrt(4*t(n)*t(n)+1) +1))

Prop.add(assumption_t0)
Prop.add(t_def)




def t_rel(n):
  return Equal( t(n+1)*t(n+1) - t(n+1), t(n)*t(n) )

proof_t0 = []
#@proof_t0.append(Eq)

#induction_eq(t_rel,[],[])

In [76]:
#Prove Case0: 0 < t(0)

In [77]:
#assume t(us) >0

In [78]:
#sum of t()

In [79]:
isinstance(t(k)*t(k),Scalar)


True

In [331]:
tnsucc = 1/2*(Sqrt(4*t(0)*t(0)+1) +1)
print(tnsucc*tnsucc) #.simplify()가 안돼여 ㅠㅠ

0.25 * (1+(1+4 * t^{0}**{2})**{0.5})**{2}


## (Temporary) strengthen verify

In [81]:
def verify_and(props, proofs):
    result = True
    for i in range(len(props.prop_list)):
        result = result and verify(props.prop_list[i], proofs[i])
    return result

#def verify_ifthen(prop, proof):
def verify_forall_scalar(prop,proof):
    arb_num = Scalar('arb_num')  #represents arbitrary number and vector
    return verify(prop(arb_num), proof)

def verify_forall_vector(prop,proof):
    arb_vec = Vector('arb_vec')
    return verify(prop(arb_vec), proof)


def verify(prop, proof):
    if callable(prop):
        try: return verify_forall_scalar(prop,proof)
        except:
            try: return verify_forall_vector(prop,proof)
            except: raise TypeError(f'wrong input type: {type(prop)}')

    if isinstance(prop, Equal):
        return verify_equality(prop, proof)
    if isinstance(prop, le):
        return verify_le(prop, proof)
    if isinstance(prop, lt):
        return verify_lt(prop, proof)
    if isinstance(prop, prop_and ):
        return verify_and(prop, proof)






In [82]:
def cube_succ(n):
  return Equal((n+1)*(n+1)*(n+1), n*n*n + 3*n*n + 3*n + 1)

def square_succ(n):
  return Equal((n+1)*(n+1), n*n +  3*n + 1)

test_and = prop_and([Equal(1,1),le(1,2),cube_succ])

verify(test_and,[[],[],[]])



True

In [83]:
arb_n = Scalar('arb_n') #arbitrary n

def induction(prop, proof_0, proof_succ):
    target = [prop]

    if isinstance(prop,prop_and): #for now we just assume that every target props are of fucntion type
        target = prop.prop_list
    #if isinstance(ifthen)

    #Case n=0
    case_0 = []
    for prop in target:
        if not callable(prop):
            raise TypeError('We only accept for all ~ in induction.')
        case_0.append(prop(0))

    case_0_istrue = verify(prop_and(case_0),proof_0)
    print(f'case 0 : {case_0_istrue} ')

    #Case succ
    case_succ = []
    for prop in target:
        Prop.add(prop(arb_n))
        case_succ.append(prop(arb_n + 1))

    case_succ_istrue = verify(prop_and(case_succ), proof_succ)
    print(f'case succ : {case_succ_istrue} ')

    if case_0_istrue and case_succ_istrue:
        Prop.add(prop)
    else: raise ValueError('Induction failed.')

In [ ]:
induction(prop_and([cube_succ, square_succ]),[[],[]],[[],[]])



## 07 06 안 되던거 되게끔 수정

In [233]:
g(x(k+1)).substitute(x_def(k))

g(-L^{-1} g(x^{k}) + x^{k})

In [235]:
tmp_0 =  1/t(k)
tmp_1 = 1- tmp_0
tmp_2 = (t(k)-1)/t(k)

tmp = Equal(1-1/t(k),(t(k)-1)/t(k))
verify_equality(tmp,[])

True

In [348]:
a = Scalar('a')
b = Scalar('b')
a**(1/2) * b**(1/2) 
(a*b)**(1/2)

a^{0.5} b^{0.5}

In [349]:
a**(1/2) * b**(1/2) - (a*b)**(1/2)

0

In [350]:
(a**(1/2))**2

a

In [351]:
printmd('$$' + str(tnsucc*tnsucc) + '$$')

$$0.25 * (1+(1+4 * t^{0}**{2})**{0.5})**{2}$$

In [352]:
repr(tnsucc*tnsucc)

'0.25 (1+(1+4 t^{0}^{2})^{0.5})^{2}'

In [353]:
a**(1/2) * b**(1/2) - Sqrt(a*b)

0

In [354]:
(a+b)**(1/2) - Sqrt(a+b)

0

In [357]:
Sqrt(a)

a^{0.5}

In [361]:
type(Sqrt(a))

__main__.Sqrt

In [362]:
type(Sqrt(a+b))

__main__.ScalarPow

In [363]:
Sqrt(a)*Sqrt(a)

a